In [ ]:
!pip install -q kafka-python pydantic pandas
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null

In [ ]:
!tar -xzf /content/kafka_2.13-3.6.1.tgz -C /content/
!mv /content/kafka_2.13-3.6.1 /content/kafka


gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now
mv: cannot move '/content/kafka_2.13-3.6.1' to '/content/kafka/kafka_2.13-3.6.1': Directory not empty


In [ ]:
import json, os
from collections import defaultdict

class MiniKafka:
    def __init__(self, base_path="/content/mini_kafka"):
        self.base_path = base_path
        os.makedirs(base_path, exist_ok=True)

    def produce(self, topic, message: dict):
        path = f"{self.base_path}/{topic}.jsonl"
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(message, ensure_ascii=False) + "\n")

    def consume(self, topic):
        path = f"{self.base_path}/{topic}.jsonl"
        if not os.path.exists(path):
            return []
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f]

    def create_topic(self, topic):

        open(f"{self.base_path}/{topic}.jsonl", "a").close()

mk = MiniKafka()
mk.create_topic("applicants")
mk.create_topic("applicants_dlq")

print("MiniKafka Ready")

MiniKafka Ready


In [ ]:
!/content/kafka/bin/kafka-broker-api-versions.sh \
    --bootstrap-server localhost:9092 | head -3

/bin/bash: line 1: /content/kafka/bin/kafka-broker-api-versions.sh: No such file or directory


In [ ]:
for topic in ["applicants", "applicants_dlq"]:
    !/content/kafka/bin/kafka-topics.sh --create \
        --topic {topic} --bootstrap-server localhost:9092 \
        --partitions 1 --replication-factor 1 --if-not-exists

!/content/kafka/bin/kafka-topics.sh --list --bootstrap-server localhost:9092

/bin/bash: line 1: /content/kafka/bin/kafka-topics.sh: No such file or directory
/bin/bash: line 1: /content/kafka/bin/kafka-topics.sh: No such file or directory
/bin/bash: line 1: /content/kafka/bin/kafka-topics.sh: No such file or directory


In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import List

class Applicant(BaseModel):
    applicant_id: int = Field(gt=0)
    job_id: int = Field(gt=0)
    name: str = Field(min_length=2)
    education: str
    experience_years: int = Field(ge=0, le=50)
    skills: List[str] = Field(min_length=1)

In [ ]:
from google.colab import drive
import pandas as pd, json, random

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Colab Notebooks/AI_Recruitment"

df = pd.read_csv(f"{BASE}/data/applicants.csv")
records = df.to_dict(orient="records")

for r in records:
    if isinstance(r.get("skills"), str):
        r["skills"] = [s.strip() for s in r["skills"].split(",")]

bad_records = [
    {"applicant_id": -5, "job_id": 2, "name": "Test",
     "education": "Bachelor", "experience_years": 3, "skills": ["Python"]},
    {"applicant_id": 9001, "name": "NoJob",
     "education": "Master", "experience_years": 2, "skills": ["SQL"]},
    {"applicant_id": 9002, "job_id": 1, "name": "X",
     "education": "Bachelor", "experience_years": -3, "skills": ["Vue"]},
    {"applicant_id": 9003, "job_id": 3, "name": "Empty",
     "education": "Bachelor", "experience_years": 4, "skills": []},
    {"applicant_id": "abc", "job_id": 1, "name": "BadType",
     "education": "Master", "experience_years": 2, "skills": ["Java"]},
]

all_records = records + bad_records
random.shuffle(all_records)
print(f"correct: {len(records)} | incorrect: {len(bad_records)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
correct: 8 | uncorrect: 5


In [ ]:
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode("utf-8")
)

for r in all_records:
    producer.send("applicants", r)

producer.flush()
print(f"done send {len(all_records)} Register to topic: applicants")

done send 13 Register to topic: applicants


In [ ]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    "applicants",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    consumer_timeout_ms=15000,
    value_deserializer=lambda v: json.loads(v.decode("utf-8"))
)

valid, rejected = [], []

for msg in consumer:
    raw = msg.value
    try:
        clean = Applicant(**raw)
        valid.append(clean.model_dump())
    except ValidationError as e:
        reason = "; ".join(
            f"{'.'.join(map(str, err['loc']))}: {err['msg']}"
            for err in e.errors()
        )
        rejected.append({"record": raw, "rejection_reason": reason})
        producer.send("applicants_dlq",
                      {"record": raw, "rejection_reason": reason})

producer.flush()
print(f"approved: {len(valid)} | rejected: {len(rejected)}")

/tmp/ipykernel_33050/1468606660.py:3: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


approved: 32 | rejected: 20


In [ ]:
dlq_df = pd.DataFrame(rejected)
dlq_df["applicant_id"] = dlq_df["record"].apply(
    lambda r: r.get("applicant_id", "MISSING"))
print(dlq_df[["applicant_id", "rejection_reason"]].to_string(index=False))

applicant_id                                                                                             rejection_reason
        9003                                             skills: List should have at least 1 item after validation, not 0
          -5                                                                 applicant_id: Input should be greater than 0
        9002 name: String should have at least 2 characters; experience_years: Input should be greater than or equal to 0
        9001                                                                                       job_id: Field required
         abc                          applicant_id: Input should be a valid integer, unable to parse string as an integer
         abc                          applicant_id: Input should be a valid integer, unable to parse string as an integer
        9003                                             skills: List should have at least 1 item after validation, not 0
        9002 name: Strin

In [ ]:
os.makedirs(f"{BASE}/output", exist_ok=True)

pd.DataFrame(valid).to_json(
    f"{BASE}/output/bronze_applicants.json",
    orient="records", force_ascii=False)

dlq_df.to_json(
    f"{BASE}/output/quarantine.json",
    orient="records", force_ascii=False)

print("Saved")


Saved
